# AttentionBench — Kaggle Benchmark Evaluation

**Can models both filter distractions AND maintain focus over extended tasks?**

Self-contained notebook for running on Kaggle with the kaggle-benchmarks SDK.

In [ ]:
import subprocess, os, sys, json, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Install kaggle-benchmarks SDK from source
if not os.path.exists("kaggle-benchmarks"):
    subprocess.run(["git", "clone", "https://github.com/Kaggle/kaggle-benchmarks.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "./kaggle-benchmarks"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf"], check=True)

# Refresh sys.path so newly installed package is importable
import site
site.addsitedir(site.getusersitepackages())
for p in site.getsitepackages():
    site.addsitedir(p)

import kaggle_benchmarks as kbench

# Clone repo and generate dataset
if not os.path.exists("attention-bench"):
    subprocess.run(["git", "clone", "https://github.com/42euge/attention-bench.git"], check=True)
os.chdir("attention-bench")
sys.path.insert(0, "src")

if not os.path.exists("data/signal_in_noise.json"):
    subprocess.run([sys.executable, "src/generate.py"], check=True)

with open("data/signal_in_noise.json") as f:
    sin_data = json.load(f)
with open("data/vigilance.json") as f:
    vig_data = json.load(f)

sin_df = pd.DataFrame(sin_data)
vig_df = pd.DataFrame(vig_data)
print(f"Loaded {len(sin_df)} SIN + {len(vig_df)} vigilance items")

In [ ]:
# Helper functions
def parse_numbered_answers(response: str, count: int) -> list[str]:
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        cleaned = re.sub(r"^\d+[\.\)\:]\s*", "", line)
        if cleaned:
            answers.append(cleaned)
    while len(answers) < count:
        answers.append("")
    return answers[:count]

def check_answer(expected: str, actual: str) -> bool:
    exp = expected.lower().strip()
    act = actual.lower().strip()
    if exp == act or exp in act:
        return True
    exp_core = re.sub(r"^(approximately|about|roughly|around)\s+", "", exp)
    if exp_core in act:
        return True
    exp_d = re.sub(r"[,\s]", "", exp)
    act_d = re.sub(r"[,\s]", "", act)
    return exp_d == act_d and exp_d != ""

## SDK Task Definitions

In [ ]:
@kbench.task(name="attention_selective")
def attention_selective(llm, prompt: str, answers: str, num_questions: int) -> tuple[int, int]:
    """Signal-in-Noise: can the model find answers in a passage buried in noise?"""
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    num_q = int(num_questions)
    parsed = parse_numbered_answers(response, num_q)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, num_q)

@kbench.task(name="attention_sustained")
def attention_sustained(llm, prompt: str, answers: str, num_subtasks: int) -> tuple[int, int]:
    """Vigilance Decrement: does accuracy decay over 100 repeated sub-tasks?"""
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    num_sub = int(num_subtasks)
    parsed = parse_numbered_answers(response, num_sub)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, num_sub)

## Run Evaluation

Configure models and run. Adjust `MODELS` and subsetting as needed.

In [ ]:
MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4",
]
MAX_ITEMS_PER_CONFIG = 3   # per (noise_type, noise_ratio). 3 = 54 items, 10 = full 180
SKIP_HIGH_RATIOS = True    # Skip 50:1 and 100:1 to save tokens

### Signal-in-Noise Evaluation

In [ ]:
%%time
# Prepare evaluation subset
sin_eval = sin_df.copy()
if SKIP_HIGH_RATIOS:
    sin_eval = sin_eval[sin_eval["noise_ratio"] <= 25]
sin_eval = sin_eval.groupby(["noise_type", "noise_ratio"]).head(MAX_ITEMS_PER_CONFIG).reset_index(drop=True)

# Prepare DataFrame for SDK
sin_eval["answers_json"] = sin_eval["answers"].apply(json.dumps)
eval_df = sin_eval[["prompt", "num_questions"]].copy()
eval_df["answers"] = sin_eval["answers_json"]

print(f"Evaluating {len(eval_df)} SIN items × {len(MODELS)} models = {len(eval_df) * len(MODELS)} LLM calls")

llms = [kbench.llms[m] for m in MODELS]
sin_runs = attention_selective.evaluate(llm=llms, evaluation_data=eval_df, n_jobs=2)
sin_results_df = sin_runs.as_dataframe()

# Merge back metadata
sin_results_df = sin_results_df.merge(
    sin_eval[["prompt", "noise_type", "noise_ratio", "passage_id", "domain", "task_id"]],
    on="prompt", how="left"
)
print(f"Collected {len(sin_results_df)} results")

In [ ]:
# SIN: Accuracy curves by noise ratio
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, ntype in zip(axes, ["unrelated", "related", "adversarial"]):
    subset = sin_results_df[sin_results_df["noise_type"] == ntype]
    for model_name in MODELS:
        m = subset[subset["model"] == model_name] if "model" in subset.columns else subset
        curve = m.groupby("noise_ratio")["result"].mean()
        ax.plot(curve.index, curve.values, "o-", label=model_name.split("/")[-1], linewidth=2)
    ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("Noise ratio")
    ax.set_title(f"{ntype}")
    ax.set_xticks(sorted(subset["noise_ratio"].unique()))
    ax.set_xticklabels(sorted(subset["noise_ratio"].unique()))
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Accuracy")
fig.suptitle("Signal-in-Noise: Accuracy vs Noise Ratio", fontsize=14)
plt.tight_layout()
plt.show()

### Vigilance Decrement Evaluation

In [ ]:
%%time
# Run vigilance evaluation — all 6 items (small, fast)
vig_eval = vig_df.copy()
vig_eval["answers_json"] = vig_eval["answers"].apply(json.dumps)
eval_df_vig = vig_eval[["prompt", "num_subtasks"]].copy()
eval_df_vig["answers"] = vig_eval["answers_json"]

print(f"Evaluating {len(eval_df_vig)} vigilance items × {len(MODELS)} models")

llms = [kbench.llms[m] for m in MODELS]
vig_runs = attention_sustained.evaluate(llm=llms, evaluation_data=eval_df_vig, n_jobs=2)
vig_results_df = vig_runs.as_dataframe()

vig_results_df = vig_results_df.merge(
    vig_eval[["prompt", "task_type", "variant", "task_id", "oddball_position"]],
    on="prompt", how="left"
)
print(f"Collected {len(vig_results_df)} results")

In [ ]:
# Vigilance: overall accuracy by task type
print("Vigilance Results:")
print(vig_results_df.groupby(["task_type", "variant"])["result"].mean().to_string())

# Summary table
print("\n\nSIN Attention Thresholds (highest ratio with ≥80% accuracy):")
for model_name in MODELS:
    for ntype in ["unrelated", "related", "adversarial"]:
        subset = sin_results_df[
            (sin_results_df.get("model", "") == model_name) &
            (sin_results_df["noise_type"] == ntype)
        ] if "model" in sin_results_df.columns else sin_results_df[sin_results_df["noise_type"] == ntype]
        if len(subset) == 0:
            continue
        curve = subset.groupby("noise_ratio")["result"].mean()
        threshold = 0
        for ratio in sorted(curve.index):
            if curve[ratio] >= 0.8:
                threshold = ratio
        print(f"  {model_name.split('/')[-1]:20s} | {ntype:12s} | threshold = {threshold}:1")

## Save Results

In [ ]:
os.makedirs("results", exist_ok=True)
sin_results_df.to_json("results/sin_results.json", orient="records", indent=2)
vig_results_df.to_json("results/vig_results.json", orient="records", indent=2)
print("Results saved to results/")